In [2]:
%reload_ext autoreload
%autoreload 2
from rewardgym.tasks.yaml_tools import load_task_from_yaml
import rewardgym

c:\Users\simon\Desktop\rewardGym\.conda\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [3]:
meta, graph, rewards = load_task_from_yaml(
    "rewardgym/tasks/template/task.yaml", rewardgym.REWARD_CLASSES
)

In [4]:
env = rewardgym.environments.base_env.BaseEnv(
    environment_graph=graph, reward_locations=rewards
)

In [5]:
rewards[1].p

[0.8, 0.2]

In [6]:
env.rng

Generator(PCG64) at 0x285279110E0

In [7]:
from rewardgym.agents.base_agent import QAgent

agent = QAgent(
    learning_rate=0.1,
    temperature=0.2,
    action_space=env.n_actions,
    state_space=env.n_states,
)


n_episodes = 1000

for t in range(n_episodes):
    obs, info = env.reset()

    done = False

    while not done:
        action = agent.get_action(obs)

        next_obs, reward, terminated, truncated, info = env.step(action)

        agent.update(obs, action, reward, terminated, next_obs)
        done = terminated or truncated
        obs = next_obs

In [8]:
# classes["ActionStimulusTooEarly"]()

In [9]:
from rewardgym.display.display_base import BaseStimulus
from typing import Tuple


class TextStimulusSimulation(BaseStimulus):
    """
    A stimulus class for text display in psychopy.
    """

    def __init__(
        self,
        label: str,
        text: str,
        position: Tuple[int, int] = None,
        name: str = "TextStimulus",
        text_color: str = "white",
        visible: bool = True,
        autodraw: bool = False,
    ):
        """
        Stimulus class for text displays.

        Parameters
        ----------
        duration : float
            Duration of the stimulus presentation.
        text : str
            The text that should be displayed on the screen.
        position : Tuple[int, int], optional
            Where to display the text (by default in px), by default None
        name : str, optional
            name of the object, will be used for logging, by default None
        text_color : str, optional
            Color of the text string, by default "white"
        """

        super().__init__(name=name, label=label, visible=visible, autodraw=autodraw)

        self.text = text
        self.position = position
        self.text_color = text_color

    def _backend_setup(self, **kwargs):
        """
        Implement this method to create the actual backend stimulus object.
        """
        pass

        self.textStim = "textstim"

    def _backend_draw(self):
        """
        Call the backend's draw method.
        """
        pass

    def _backend_update(self):
        """
        Call the backend's draw method.
        """
        pass

In [10]:
import numpy as np

In [11]:
class BasePeriod:
    def __init__(self, name, label, onset, duration, **kwargs):
        self.name = name
        self.label = label
        self.onset = onset
        self.duration = duration
        self.started = False
        self.ended = False
        self.rllabel = None
        self.kwargs = kwargs

    def start(self, t, episode, **kwargs):
        if not self.started and check_timing(self.onset, t, episode, 0):
            self.started = True
            self.start_time = t

    def end(self, t, episode, **kwargs):
        if self.started and check_timing(self.duration, t, episode, self.start_time):
            if not self.ended:
                self.ended = True

    def update(self, t, **kwargs):
        s = self.start(t, **kwargs)
        h = self.handle(t, **kwargs)
        e = self.end(t, **kwargs)

        for tmp in [s, h, e]:
            if tmp is not None:
                return tmp

    def handle(self, t, **kwargs):
        # To be implemented in subclasses
        pass

    def reset(self):
        self.ended = False
        self.started = False


class ActionPeriod(BasePeriod):
    def __init__(
        self,
        name,
        label,
        onset,
        duration,
        timeout_action,
        timeout_name,
        key_dict,
        stop_after_response,
        **kwargs,
    ):
        self.name = name
        self.label = label
        self.onset = onset
        self.duration = duration
        self.started = False
        self.ended = False
        self.rllabel = None
        self.response = False
        self.kwargs = kwargs
        self.timeout_action = timeout_action
        self.timeout_name = timeout_name
        self.key_dict = key_dict
        self.key_list = list(key_dict.keys())
        self.response = False
        self.action = None
        self.rt = None
        self.stop_after_response = stop_after_response

    def handle(self, t, presses, env, **kwargs):
        # To be implemented in subclasses

        if self.started and not self.ended:
            for n, resp in enumerate(presses):
                if resp[0] in self.key_list:
                    response = (resp[0], resp[1])
                    presses.pop(n)

                    response_time = response[1] - self.start_time
                    response_button = response[0]

                    if env.info is not None and "behav_remap" in env.info.keys():
                        action = env.info["behav_remap"][self.key_dict[response[0]]]
                    else:
                        action = self.key_dict[response[0]]

                    self.response = True
                    self.action = action

                    if self.stop_after_response:
                        self.ended = True

                    return {
                        "logger": {
                            "info_dict": {
                                "event_type": self.label,
                                "response_button": response[0],
                                "response_time": response_time,
                                "action": action,
                                "rllabel": self.rllabel,
                            },
                            "onset": self.start_time,
                        },
                    }

    def end(self, t, episode, **kwargs):
        if self.started and check_timing(self.duration, t, episode, self.start_time):
            if not self.ended:
                self.ended = True

                if not self.response:
                    if self.timout_action is not None:
                        self.action = action
                    return {
                        "logger": {
                            "info_dict": {
                                "event_type": self.timeout_name,
                                "response_late": True,
                                "response_time": None,
                                "response_button": None,
                                "action": self.timeout_action,
                                "rllabel": self.rllabel,
                            },
                            "onset": self.start_time,
                        }
                    }


class EventPeriod(BasePeriod):
    def end(self, t, episode, **kwargs):
        if self.started and check_timing(self.duration, t, episode, self.start_time):
            if not self.ended:
                self.ended = True
                print(t, self.start_time)
                return {
                    "logger": {
                        "info_dict": {
                            "event_type": self.label,
                            "expected_duration": self.duration,
                            "rllabel": self.rllabel,
                        },
                        "onset": self.start_time,
                    }
                }

In [12]:
from rewardgym.tasks.yaml_tools import load_yaml
from rewardgym.display.psychopy.psychopy_stubs import Clock, Window
from rewardgym.display.new_logger import SimulationLogger

test_dict = load_yaml("rewardgym/tasks/template/display.yaml")

In [13]:
framerate = 1 / 32
episode = test_dict["templates"][test_dict["episodes"][0]["use"]]
stimuli = {
    "fixation": TextStimulusSimulation(
        label="fixation", text="+", position=[0, 0], text_color="white"
    ),
    "stim_left": TextStimulusSimulation(
        label="stim_left", text="A", position=[-200, 0], text_color="white"
    ),
    "stim_right": TextStimulusSimulation(
        label="stim_right", text="B", position=[200, 0], text_color="white"
    ),
}

In [14]:
episode["periods"] = [
    EventPeriod("base", label="fix", rllabel="obs", onset=0.0, duration=0.5),
    ActionPeriod(
        name="response",
        label="response",
        rllable="action",
        onset=0.5,
        duration=2.5,
        timeout_action=1,
        timeout_name="response_late",
        key_dict={"space": 0},
        stop_after_response=False,
    ),
]

In [20]:
episode['hooks'] = [
    

]

{'duration': 3.0,
 'render': [{'stim': 'fixation', 'onset': 0.0, 'duration': 0.5},
  {'stim': 'stim_left', 'onset': 0.5, 'duration': 'periods.response.ended'},
  {'stim': 'stim_right', 'onset': 0.5, 'duration': 2.5}],
 'periods': [<__main__.EventPeriod at 0x2852797b970>,
  <__main__.ActionPeriod at 0x2852797bb20>]}

In [16]:
episode["render"][1]["duration"] = "periods.response.ended"

In [17]:
class EpisodeHandler:
    def __init__(self):
        pass

In [19]:
from rewardgym.display.handling import attribute_getter, attribute_setter, trigger_handler
from typing import Dict, List, Union

class TestHook:
    def __init__(self):
        pass

    def update(self, **kwargs):
        pass


class UpdateStimulus(TestHook):
    def __init__(self, target: Union[List[str], str], 
                 trigger: str, update_value: Dict, 
                 entry_point, 
                 reset_entry=False):
        self.target = target
        self.entry_point = entry_point
        self.trigger = trigger
        self.reset_entry = reset_entry
        self.update_value = update_value
        self.old_value = None

        if self.reset_entry:
            self.old_value = {k : None for k in self.update_value.keys()}

    def update(entry, t,  episode):
        if entry == self.entry_point:
            if trigger_handler(self.trigger, episode) is not None:

                for k, v in self.update_value:

                    if self.reset_entry and self.old_value[k] is None:
                        self.old_value[k] = attribute_getter(episode, self.target, k)

                    attribute_setter(episode, self.target, k, v)


In [54]:
from rewardgym.display.handling import check_timing


logger = SimulationLogger(
    file_name="abc",
    global_clock=Clock(),
    participant_id="n/a",
    task="hcp",
    run=1,
    seq_tr=0.752,
    sep="\t",
    na="n/a",
)

clock = Clock()
win = Window()
start_time = clock.getTime()
logger.create()
logger.global_clock.reset()
logger.set_trial_time()

framerate = 60


while not check_timing(
    episode["duration"], clock.getTime(), episode
):  # clock.getTime() < episode["duration"]:
    t = clock.getTime() - start_time

    events = []

    presses = []
    # constant polling of keyboard buffer
    # presses = getKeys(timeStamped=self.global_clock)

    if np.isclose(t, 1.5):
        presses.append(("space", 1.5))

    for rend in episode["render"]:
        stimuli[rend["stim"]].draw(
            current_time=t,
            onset=rend["onset"],
            duration=rend["duration"],
            episode=episode,
        )

    for period in episode["periods"]:
        op = period.update(t, presses=presses, env=env, episode=episode)
        if op is not None:
            events.append(op)

    for op in events:
        if "logger" in op.keys():
            print(op)
            logger.log_event(**op["logger"])

    win.flip()
    clock.time += 1 / framerate
    logger.global_clock.time += 1 / framerate

    for op in events:
        if "terminated" in op.keys():
            if op["terminated"]:
                break


for period in episode["periods"]:
    period.reset()

print(clock.time)

logger.close()

0.5333333333333333 0.016666666666666666
{'logger': {'info_dict': {'event_type': 'fix', 'expected_duration': 0.5, 'rllabel': None}, 'onset': 0.016666666666666666}}
{'logger': {'info_dict': {'event_type': 'response', 'response_button': 'space', 'response_time': 0.9833333333333334, 'action': 0, 'rllabel': None}, 'onset': 0.5166666666666666}}
3.016666666666661


{'onset': ['0.016666666666666666', '0.5166666666666666'],
 'duration': ['0.5166666666666666', '0.983333333333333'],
 'trial_type': ['n/a', 'n/a'],
 'event_type': ['fix', 'response'],
 'response_time': ['n/a', '0.9833333333333334'],
 'response_button': ['n/a', 'space'],
 'response_late': ['n/a', 'n/a'],
 'response_ignore': ['n/a', 'n/a'],
 'action': ['n/a', '0'],
 'reward': ['None', 'None'],
 'trial': ['-1', '-1'],
 'rllabel': ['n/a', 'n/a'],
 'current_location': ['n/a', 'n/a'],
 'trial_time': ['0.5333333333333333', '1.4999999999999996'],
 'total_reward': ['n/a', 'n/a'],
 'avail_actions': ['n/a', 'n/a'],
 'misc': ['n/a', 'n/a'],
 'TR': ['0', '0'],
 'expected_duration': ['0.5', 'n/a'],
 'start_position': ['n/a', 'n/a'],
 'task': ['n/a', 'n/a'],
 'run': ['1', '1'],
 'participant_id': ['n/a', 'n/a']}

In [155]:
from rewardgym.tasks.yaml_tools import load_yaml

In [165]:
t

3.0333333333333274

{'onset': ['0.5166666666666666'],
 'duration': ['n/a'],
 'trial_type': ['n/a'],
 'event_type': ['fix'],
 'response_time': ['n/a'],
 'response_button': ['n/a'],
 'response_late': ['n/a'],
 'response_ignore': ['n/a'],
 'action': ['n/a'],
 'reward': ['None'],
 'trial': ['-1'],
 'rllabel': ['n/a'],
 'current_location': ['n/a'],
 'trial_time': ['0.5166666666666666'],
 'total_reward': ['n/a'],
 'avail_actions': ['n/a'],
 'misc': ['n/a'],
 'TR': ['0'],
 'expected_duration': ['0.5'],
 'start_position': ['n/a'],
 'task': ['n/a'],
 'run': ['1'],
 'participant_id': ['n/a']}

In [50]:
test_dict = load_yaml("rewardgym/tasks/template/display.yaml")

In [52]:
test_dict["episodes"][0]
test_dict["templates"][test_dict["episodes"][0]["use"]]

{'duration': 3.0,
 'render': [{'stim': 'fixation', 'onset': 0.0, 'duration': 0.5},
  {'stim': 'stim_left', 'onset': 0.5, 'duration': 2.5},
  {'stim': 'stim_right', 'onset': 0.5, 'duration': 2.5}],
 'periods': [{'name': 'base',
   'onset': 0.0,
   'duration': 0.5,
   'label': 'fixation',
   'rllabel': 'obs'},
  {'name': 'response',
   'onset': 0.5,
   'duration': 2.5,
   'listen_for_keys': ['left', 'right'],
   'label': 'response'}]}

In [53]:
test_dict["stimuli"]

{'fixation': {'type': 'TextStimulus',
  'text': '+',
  'pos': [0, 0],
  'color': 'white'},
 'stim_left': {'type': 'TextStimulus',
  'text': 'A',
  'pos': [-200, 0],
  'color': 'white'},
 'stim_right': {'type': 'TextStimulus',
  'text': 'B',
  'pos': [200, 0],
  'color': 'white'},
 'reward': {'type': 'TextStimulus',
  'text': '+',
  'pos': [0, 50],
  'color': 'white'}}

In [46]:
setattr(episode["periods"][1], "started", 1)

In [42]:
a = True

if a:
    print("a")

a


In [47]:
episode["periods"][1].started

1